# Appendix: How Language Models Generate Text

A language model repeatedly predicts a distribution for the next token, selects one token, and continues until it stops.


## What to learn

- Tokens are pieces of text, not necessarily complete words.
- Attention mixes information from earlier tokens to build each prediction.
- The KV cache avoids recomputing unchanged attention state during generation.
- Sampling settings change variability but do not add knowledge or reasoning.

## Decision rule

Use this model to reason about latency, context limits, and variability. Avoid assuming that generated fluency implies stored facts, deterministic behavior, or human-like internal reasoning.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
"""The generation loop, faithfully miniaturized: tokens -> logits ->
temperature/top-p sampling -> append -> repeat, with a toy KV cache
demonstrating why prefix reuse saves money.
"""
# Import the dependencies used by this example.
import math
import random

VOCAB = ["overdue", "paid", "pending", "banana"]


# Define the data shape and small operations before running them.
def fake_logits(context_tokens):
    """Stands in for N transformer blocks: context -> score per vocab token."""
    base = {"overdue": 3.0, "paid": 2.2, "pending": 1.5, "banana": -2.0}
    return [base[t] for t in VOCAB]


def sample(logits, temperature=0.8, top_p=0.9, rng=None):
    if temperature == 0:
        return VOCAB[logits.index(max(logits))]
    scaled = [l / temperature for l in logits]
    exps = [math.exp(l - max(scaled)) for l in scaled]
    probs = [e / sum(exps) for e in exps]
    # top-p: keep the smallest set of tokens with cumulative prob >= p
    order = sorted(range(len(VOCAB)), key=lambda i: -probs[i])
    kept, cum = [], 0.0
    for i in order:
        kept.append(i); cum += probs[i]
        if cum >= top_p:
            break
    kept_probs = [probs[i] for i in kept]
    total = sum(kept_probs)
    return VOCAB[rng.choices(kept, [p / total for p in kept_probs])[0]]


rng = random.Random(0)
for temp in (0, 0.8, 2.0):
    outs = [sample(fake_logits(["the", "invoice", "is"]), temp, rng=rng or random.Random())
            if temp else sample(fake_logits([]), 0) for _ in range(8)]
    print(f"T={temp}: {outs}")
# T=0 always argmax; T=0.8 mostly 'overdue'; T=2.0 nearly uniform over kept set.

# --- KV cache economics ----------------------------------------------------
def cost_without_cache(prompt_len, gen_len):
    # every step reprocesses the whole prefix
    return sum(prompt_len + i for i in range(gen_len))

def cost_with_cache(prompt_len, gen_len, prefix_cached=0):
    prefill = prompt_len - prefix_cached      # only uncached prefix prefilled
    return prefill + gen_len                  # then one unit per new token

P, G = 5000, 200
print(f"\nno KV cache:          {cost_without_cache(P, G):>9,} token-passes")
print(f"KV cache:             {cost_with_cache(P, G):>9,}")
print(f"KV + prompt caching:  {cost_with_cache(P, G, prefix_cached=4500):>9,}"
      "  <- why stable prefixes cut cost 50-90%")


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
